In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import time
import datetime
import random
import cv2
import warnings
import math
from tabulate import tabulate
from skimage.morphology import skeletonize
from tqdm import tqdm
from skimage import io, color, morphology
from scipy.ndimage import distance_transform_edt
import scipy.ndimage.morphology
import scipy.io as sio
sys.path.append('../')  # Adds the root directory to Python's module search path
from MLVcode.load_mat import load_mat
from MLVcode.unique import unique1D, unique2D, unique3D

In [ ]:
data_keys = {'originalImage','imsize', 'lineMethod', 'numContours',                
            'contours', 'orientations', 'lengths',   'contourLengths',             
            'curvatures', 'junctions', 'parallelism',  'parallelismMeans',           
            'parallelism_allX', 'parallelism_allY','parallelism_allScores', 
            'parallelismBins', 'parallelismHistograms', 'parallelismNormHistograms',  
            'parallelismSumHistogram', 'parallelismNormSumHistogram',
            'separation', 'separationMeans', 'separation_allX',  'separation_allY',            
            'separation_allScores', 'separationBins',             
            'separationHistograms', 'separationNormHistograms',   
            'separationSumHistogram', 'separationNormSumHistogram', 
            'mirror', 'mirrorMeans', 'mirror_allX', 'mirror_allY',                
            'mirror_allScores', 'mirrorBins', 'mirrorHistograms', 
            'mirrorNormHistograms', 'mirrorSumHistogram', 'mirrorNormSumHistogram'}

# filename='../dataSets/TorontoScenes/mountains_vecLD.mat'
filename = '../dataSets/TorontoScenes/mountains_vecLD.mat'
data = load_mat(filename)

In [ ]:
# Note that temporarily, rendering function is yet to be implemented in MATLAB.
# So there is need to obtained imgLD from MATLAB just for this functionality temporarily until the rendering function is fully complete

In [ ]:
# def renderLinedrawingImg(vecLD):
#     """
#     Obtain scaled drawings.
#     Parameters:
#         vecLD (dict): The vectorized line drawing to be drawn. Expected to have 'contours' and 'imsize'.
#         img (PIL.Image): An existing PIL image to draw into. If None, a new RGB image with a white background
#                          will be initialized.
#         imsize (tuple): The image size in (width, height). If imsize is different from vecLD['imsize'],
#                         the drawing will be scaled up or down to the new imsize. If None, use vecLD['imsize'].
#         lineWidth (int): The width of the contour lines in pixels. Default is 1.
#         color (tuple): The RGB color for drawing the contours. Default is (0, 0, 0) which is black.

#     Returns:
#         Image: The RGB image coordinates with the contours drawn in.
#     """
#     imsize = vecLD[1][0]
#     # Calculate scale factors
#     scale_x = imsize[0] / vecLD['imsize'][0]
#     scale_y = imsize[1] / vecLD['imsize'][1]

#     # Loop over contours and draw each one
#     for contour in vecLD[4][0]:
#         # Scale the coordinates
#         scaled_coords = [(x * scale_x, y * scale_y) for x, y in contour]


#     return img

# data['vecLD'][0][10][1][0]
# data['vecLD'][0][10][4][0][1]

In [ ]:
cute = {}
# Assign the data to the dictionary and define them with empty value
for key in data_keys:
    cute[key] = data['vecLD'][key][0][10]
print("Name: ", cute['originalImage'])
print("Shape of contour: ", cute['contours'].shape)
print("Size of image: ", cute['imsize'])
# Load imgLD.mat in same folder
imgLD = load_mat('imgLD.mat') # Need to get from MATLAB temporarily until the rendering function capablity is added to the Python version.
# imgLD = load_mat(filename)
# imgLD = imgLD['vecLD'].copy()
# Convert 1s to 255s
# make a copy of the array
temp = imgLD['imgLD'].copy()
# Replace 1s with 255s
temp[temp==1] = 255
uni, count = unique3D(imgLD['imgLD'])
# Print the unique values and their counts
print(tabulate({'Unique':uni, 'Count':count}, headers='keys', tablefmt='psql'))
print("Unique values in the 3D array: ", uni.shape[0], "the values are:\n", uni)
print("Counts of unique values in the 3D array: ", count.shape[0], "the counts are: \n", count)
plt.imshow(temp)
plt.show()

In [ ]:
from MLVcode.bwdist import bwdist
from MLVcode.is_outer_border_point import is_outer_border_point
from MLVcode.getOuterBoundary import getOuterBoundary
from MLVcode.computeGradientVectorField import computeGradientVectorField
from MLVcode.sample_sphere_2D import sample_sphere_2D
from MLVcode.computeAOF import computeAOF
from MLVcode.extract2DSkeletonFromBinaryImage import extract2DSkeletonFromBinaryImage
from MLVcode.computeMAT import computeMAT

In [ ]:
# imgLD['imgLD']
test = imgLD['imgLD'].copy()
# Print the shape of the image
print("Shape of image: ", test.shape)
# make it 2D instead of 3D
test = test[:,:,0]
# Skeletonize the image
test = ~test.astype(bool)
test = skeletonize(test, method='lee').astype(np.uint8)
# Invert the image
test = cv2.bitwise_not(test)
# Invert the image using bitwise_not
plt.imshow(test, cmap='gray')
# title
plt.title("Skeletonized Image")
plt.show()

In [ ]:
# Compute Medial Axis - slower than the MATLAB version - scope for optimisation
mat, fluxImage, skeletonImage, distImage = computeMAT(imgLD['imgLD'])

In [ ]:
# Save fluxImage as a .mat file
sio.savemat('fluxImage_temp.mat', {'fluxImage':fluxImage})

In [ ]:
orange_skeleton = np.stack([skeletonImage, 0.5 * skeletonImage, np.zeros_like(skeletonImage)], axis=-1)
# Plot the image
plt.figure()
invert = ~(imgLD['imgLD']*255)
invert = invert+orange_skeleton
plt.imshow(invert, cmap='gray')
plt.title('Skeleton')
plt.show()

In [ ]:
plt.figure()
plt.imshow(distImage)
plt.imshow((imgLD['imgLD']*255), alpha=0.2)
plt.title("Distance Map")
cbar = plt.colorbar()  # Create a colorbar and keep a reference to the Colorbar object
cbar.set_label("Distance to the closest contour")
cbar.mappable.set_cmap('viridis')
plt.show()

In [ ]:
# load_mat('fluxImage.mat')['fluxImage']
plt.figure()
plt.imshow(load_mat('fluxImage_temp.mat')['fluxImage'], cmap='gray')
plt.imshow(~(imgLD['imgLD']*255), alpha=0.01, cmap='Reds')
# Add title
plt.title("Average Outward Flux")
plt.show()

In [ ]:
# Making it look better
plt.figure()
from matplotlib.colors import LinearSegmentedColormap
bw_cmap = LinearSegmentedColormap.from_list('bw_cmap', [(0, 'black'), (1, 'white')])
cmap = LinearSegmentedColormap.from_list('mycmap', [(0, (1, 0, 0, 0)), (1, (1, 0, 0, 1))])
fluxImage = load_mat('fluxImage_temp.mat')['fluxImage']
img_LD = imgLD['imgLD']


plt.figure()
plt.subplot(1, 2, 1)
# Plot the original image
plt.imshow(img_LD*255, cmap='gray')
plt.title("Original Image")
plt.subplot(1, 2, 2)
# Plot the fluxImage
plt.imshow(fluxImage, cmap=bw_cmap, vmin=0, vmax=10, alpha=1.0)
plt.imshow((img_LD*255),cmap=cmap, vmin=5, vmax=10, alpha=0.2)
plt.title("Average Outward Flux Map")
plt.show()